In [ ]:
# Install playwright, its system dependencies, and the chromium browser
!pip install playwright
!playwright install-deps chromium
!playwright install chromium

In [5]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import pandas as pd
import sys
import os

# --- GOOGLE DRIVE INTEGRATION ---
# When running in Colab, this will prompt you to authorize access to your Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Define a path in your Drive. Change 'My Drive/ScraperData' to your preferred folder.
    SAVE_DIRECTORY = '/content/drive/My Drive/PropertyFinder_Scrapes'
    if not os.path.exists(SAVE_DIRECTORY):
        os.makedirs(SAVE_DIRECTORY)
    SAVE_PATH = os.path.join(SAVE_DIRECTORY, 'property_finder_listings.csv')
    print(f"Persistent storage enabled. Files will save to: {SAVE_PATH}")
except ImportError:
    # Fallback for local execution
    SAVE_PATH = 'property_finder_listings.csv'
    print("Running locally. File will save to current directory.")

# Base URL for Property Finder Egypt
BASE_URL = "https://www.propertyfinder.eg/en/search?c=1&fu=0&ob=mr"

async def get_page_html(browser_context, page_number):
    """
    Navigates to a specific page and returns the HTML without scrolling.
    """
    page = await browser_context.new_page()
    page_url = f"{BASE_URL}&page={page_number}"

    print(f"\n--- Navigating to Page {page_number}: {page_url} ---")
    try:
        await page.goto(page_url, wait_until="networkidle", timeout=60000)
        html_content = await page.content()
        await page.close()
        return html_content
    except Exception as e:
        print(f"Error loading page {page_number}: {e}")
        await page.close()
        return None

def parse_property_data(html):
    """
    Parses Property Finder HTML using data-testid attributes.
    """
    if not html:
        return []

    soup = BeautifulSoup(html, 'html.parser')
    all_properties_data = []
    cards = soup.find_all('article', {'data-testid': 'property-card'})

    for card in cards:
        extracted_data = {}

        def get_by_testid(testid):
            target = card.find(attrs={"data-testid": testid})
            return target.get_text(strip=True) if target else None

        link_elem = card.find('a', {'data-testid': 'property-card-link'})
        if link_elem and 'href' in link_elem.attrs:
            full_link = link_elem['href']
            extracted_data['link'] = full_link if full_link.startswith('http') else f"https://www.propertyfinder.eg{full_link}"
        else:
            extracted_data['link'] = None

        extracted_data['title'] = card.find('h3').get_text(strip=True) if card.find('h3') else None

        img_elem = card.find('img', {'data-testid': 'gallery-picture', 'class': 'styles_desktop_gallery-image__n81d5'})
        if not img_elem:
            img_elem = card.find('img', {'data-testid': 'gallery-picture'})
        extracted_data['property_image'] = img_elem.get('src') if img_elem else None

        extracted_data['property_type'] = get_by_testid("property-card-type")
        extracted_data['price'] = get_by_testid("property-card-price")
        extracted_data['location'] = get_by_testid("property-card-location")

        extracted_data['bedrooms'] = get_by_testid("property-card-spec-bedroom")
        extracted_data['bathrooms'] = get_by_testid("property-card-spec-bathroom")
        extracted_data['area'] = get_by_testid("property-card-spec-area")
        extracted_data['price_per_sqm'] = get_by_testid("property-card-spec-price-per-area")

        broker_container = card.find(attrs={"data-testid": "property-card-broker-logo"})
        if broker_container:
            broker_img = broker_container.find('img', {'data-testid': 'gallery-picture'})
            if not broker_img:
                broker_img = broker_container.find('img')
            extracted_data['broker_logo'] = broker_img.get('src') if broker_img else None
        else:
            extracted_data['broker_logo'] = None

        all_properties_data.append(extracted_data)

    return all_properties_data

async def main():
    max_pages = 800  # You can safely increase this now that data is backed up to Drive
    all_results = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
        )

        for page_num in range(1, max_pages + 1):
            html = await get_page_html(context, page_num)
            page_data = parse_property_data(html)
            print(f"Extracted {len(page_data)} properties from page {page_num}.")
            all_results.extend(page_data)

            # OPTIONAL: Save "checkpoints" every 2 pages to prevent loss during long runs
            if page_num % 2 == 0:
                temp_df = pd.DataFrame(all_results).drop_duplicates(subset=['link'])
                temp_df.to_csv(SAVE_PATH, index=False)
                print(f"--- Checkpoint: Saved {len(temp_df)} items to Drive ---")

        await browser.close()

    if all_results:
        df = pd.DataFrame(all_results)
        df = df.drop_duplicates(subset=['link'])

        cols = [
            'title', 'price', 'location', 'property_type',
            'bedrooms', 'bathrooms', 'area',
            'broker_logo', 'property_image', 'link'
        ]
        df = df[[c for c in cols if c in df.columns]]

        # Final Save
        df.to_csv(SAVE_PATH, index=False)
        print(f"\nTotal Success! Extracted {len(df)} unique properties across {max_pages} pages.")
        print(f"Data is permanently saved to your Google Drive at: {SAVE_PATH}")
        return df
    else:
        print("No properties found.")
        return None

if __name__ == "__main__":
    try:
        loop = asyncio.get_running_loop()
        print("Notebook environment detected. Run 'await main()' in a cell to start.")
    except RuntimeError:
        asyncio.run(main())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent storage enabled. Files will save to: /content/drive/My Drive/PropertyFinder_Scrapes/property_finder_listings.csv
Notebook environment detected. Run 'await main()' in a cell to start.


In [6]:
await main()


--- Navigating to Page 1: https://www.propertyfinder.eg/en/search?c=1&fu=0&ob=mr&page=1 ---
Extracted 25 properties from page 1.

--- Navigating to Page 2: https://www.propertyfinder.eg/en/search?c=1&fu=0&ob=mr&page=2 ---
Extracted 25 properties from page 2.
--- Checkpoint: Saved 50 items to Drive ---

--- Navigating to Page 3: https://www.propertyfinder.eg/en/search?c=1&fu=0&ob=mr&page=3 ---
Extracted 25 properties from page 3.

--- Navigating to Page 4: https://www.propertyfinder.eg/en/search?c=1&fu=0&ob=mr&page=4 ---
Extracted 25 properties from page 4.
--- Checkpoint: Saved 100 items to Drive ---

--- Navigating to Page 5: https://www.propertyfinder.eg/en/search?c=1&fu=0&ob=mr&page=5 ---
Extracted 25 properties from page 5.

--- Navigating to Page 6: https://www.propertyfinder.eg/en/search?c=1&fu=0&ob=mr&page=6 ---
Extracted 25 properties from page 6.
--- Checkpoint: Saved 144 items to Drive ---

--- Navigating to Page 7: https://www.propertyfinder.eg/en/search?c=1&fu=0&ob=mr&page

,title,price,location,property_type,bedrooms,bathrooms,area,broker_logo,property_image,link
0,Super lux apartment for resale - Al-Ahly Sabour,"5,900,000 EGP","The Square, 5th Settlement Compounds, The 5th ...",Apartment,2,2,125 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/apart...
1,Apart. 3BR for resale including maintenance & ...,"8,750,000 EGP","Palm Hills New Cairo, 5th Settlement Compounds...",Apartment,3,3,183 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/apart...
2,Luxury apartment for resale in Lake View Resid...,"13,000,000 EGP","Lake View Residence, 5th Settlement Compounds,...",Apartment,2,2,144 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/apart...
3,APT Fully Finished Ready To Move Prime Location,"8,800,000 EGP","The Square, 5th Settlement Compounds, The 5th ...",Apartment,3,3,210 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/apart...
4,APT with Garden READY TO MOVE Under Market Price,"9,250,000 EGP","The Square, 5th Settlement Compounds, The 5th ...",Apartment,3,3,170 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/apart...
...,...,...,...,...,...,...,...,...,...,...
19970,"chalet for sale, Seashore Hyde Park, Sea View","20,000,000 EGP","Seashore, Ras Al Hekma, North Coast",Chalet,3,2,130 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/chale...
19971,Chalet 150m sea view and ready to move,"4,160,000 EGP","La vista Ras El Hikma, Ras Al Hekma, North Coast",Chalet,3,2,150 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/chale...
19972,First-row lagoon Twin house for sale in Azha S...,"19,000,000 EGP","Azha, Al Ain Al Sokhna, Suez",Twin House,3,3,185 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/twin-...
19973,"Ground floor chalet for sale, Seashore, Hyde Park","16,050,000 EGP","Seashore, Ras Al Hekma, North Coast",Chalet,3,3,138 sqm,https://static.shared.propertyfinder.eg/media/...,https://static.shared.propertyfinder.eg/media/...,https://www.propertyfinder.eg/en/plp/buy/chale...
